# 📘 Capstone Project: FinTech Data Platform
## Build a Compliance-Aware Banking Data Pipeline

**Scenario:** You are building the data platform for a digital bank.
Financial data requires precision, compliance, audit trails, and fraud detection.

**Deliverables:**
1. CDC processing pipeline (wire_transfers with I/U/D operations)
2. Fraud scoring engine (composite risk scores per account)
3. KYC compliance tracking (SCD Type 2)
4. Regulatory reporting Gold tables (SAR, KYC dashboard, transaction summary)
5. Alerting logic with deduplication
6. Audit trail and data retention

**Estimated time:** 6-8 hours

**Prerequisites:** Notebook 04 (fintech_dw must exist)

---

---
# 🏗️ Section 1: Architecture Design

```
┌──────────────────────────────────────────────────────────────────────┐
│              FINTECH DATA PLATFORM (Compliance-Aware)                  │
├──────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  CDC FEED              BRONZE           SILVER            GOLD         │
│  ────────              ──────           ──────            ────         │
│  wire_transfers ──▶ bronze_wt ──CDC──▶ silver_wt ──▶ gold_txn_summary│
│  (I/U/D ops)                                       ──▶ gold_sar_data │
│                                                                        │
│  fraud_signals ───▶ bronze_fraud ────▶ silver_fraud ─▶ gold_risk     │
│  kyc_records ─────▶ bronze_kyc ──────▶ dim_kyc_scd2 ─▶ gold_kyc     │
│  compliance ──────▶ bronze_comp ─────▶ silver_comp ──▶ gold_reg      │
│                                                                        │
│  CROSS-CUTTING:                                                        │
│  ├── Audit Log (all changes tracked)                                  │
│  ├── Alert Engine (threshold-based)                                   │
│  ├── Data Retention (7yr txn, 5yr fraud, lifetime+5 KYC)             │
│  └── Point-in-Time Queries (regulatory audits)                        │
└──────────────────────────────────────────────────────────────────────┘
```

## Data Classification

| Data Type | Classification | Retention | Access |
|---|---|---|---|
| Wire transfers | Financial/PII | 7 years | Restricted |
| Fraud signals | Security | 5 years | Security team |
| KYC records | PII/Regulatory | Account lifetime + 5yr | Compliance |
| Compliance events | Regulatory | 10 years | Compliance/Legal |

---
# 🔄 Section 2: CDC Processing Pipeline

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("USE fintech_dw")

# Read CDC feed from wire_transfers
cdc_feed = spark.table("wire_transfers")
print(f"CDC feed: {cdc_feed.count():,} records")
print(f"\nOperation distribution:")
cdc_feed.groupBy("_cdc_operation").count().show()

In [ ]:
%sql
-- Apply CDC: deduplicate (keep latest per transfer_id), then MERGE
USE fintech_dw;

DROP TABLE IF EXISTS capstone_silver_wire_transfers;
CREATE TABLE capstone_silver_wire_transfers AS
SELECT transfer_id, from_account, to_account, amount, currency,
  transfer_date, status, reference_number, _cdc_timestamp,
  current_timestamp() AS _processed_at
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY transfer_id ORDER BY _cdc_timestamp DESC) AS rn
  FROM wire_transfers
  WHERE _cdc_operation != 'D'
) WHERE rn = 1;

SELECT COUNT(*) AS silver_wt_count FROM capstone_silver_wire_transfers;

In [ ]:
%sql
-- Simulate incremental CDC batch (new records arriving)
-- MERGE pattern for applying CDC incrementally
DROP TABLE IF EXISTS capstone_cdc_staging;
CREATE TABLE capstone_cdc_staging AS
SELECT * FROM wire_transfers WHERE transfer_id BETWEEN 1000 AND 1100;

-- Apply CDC via MERGE
MERGE INTO capstone_silver_wire_transfers AS target
USING (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY transfer_id ORDER BY _cdc_timestamp DESC) AS rn
    FROM capstone_cdc_staging WHERE _cdc_operation IN ('I', 'U')
  ) WHERE rn = 1
) AS source
ON target.transfer_id = source.transfer_id
WHEN MATCHED AND source._cdc_timestamp > target._cdc_timestamp THEN
  UPDATE SET target.amount = source.amount, target.status = source.status,
    target._cdc_timestamp = source._cdc_timestamp, target._processed_at = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (transfer_id, from_account, to_account, amount, currency, transfer_date, status, reference_number, _cdc_timestamp, _processed_at)
  VALUES (source.transfer_id, source.from_account, source.to_account, source.amount, source.currency, source.transfer_date, source.status, source.reference_number, source._cdc_timestamp, current_timestamp());

In [ ]:
# ============================================
# 🎯 YOUR TURN: CDC for compliance_events
# ============================================
# 1. Read compliance_events from fintech_dw
# 2. Deduplicate by event_id (keep latest)
# 3. Write to capstone_silver_compliance
# 4. Print operation counts and final row count
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *
from pyspark.sql.window import Window

comp = spark.table("compliance_events")
w = Window.partitionBy("event_id").orderBy(col("event_date").desc())
silver_comp = (comp
    .withColumn("_rn", row_number().over(w))
    .filter("_rn = 1").drop("_rn")
    .withColumn("_processed_at", current_timestamp())
)
silver_comp.write.format("delta").mode("overwrite").saveAsTable("capstone_silver_compliance")
print(f"✅ Silver compliance: {spark.table('capstone_silver_compliance').count():,} rows")

---
# 🚨 Section 3: Fraud Scoring Engine

In [ ]:
# Build composite fraud risk scores per account
from pyspark.sql.functions import *

fraud = spark.table("fraud_signals")
transfers = spark.table("capstone_silver_wire_transfers")
accounts = spark.table("accounts")

# Fraud signal aggregation
fraud_agg = (fraud
    .groupBy("account_id")
    .agg(
        count("*").alias("total_signals"),
        sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_signals"),
        sum(when(col("severity") == "high", 1).otherwise(0)).alias("high_signals"),
        sum(when(col("is_confirmed") == True, 1).otherwise(0)).alias("confirmed_fraud"),
        countDistinct("signal_type").alias("unique_signal_types")
    )
)

# Transaction velocity metrics
velocity = (transfers
    .groupBy(col("from_account").alias("account_id"))
    .agg(
        count("*").alias("total_transfers"),
        round(sum("amount"), 2).alias("total_amount_sent"),
        round(avg("amount"), 2).alias("avg_transfer_amount"),
        round(max("amount"), 2).alias("max_single_transfer"),
        countDistinct("to_account").alias("unique_recipients"),
        countDistinct("transfer_date").alias("active_days")
    )
    .withColumn("avg_daily_amount", round(col("total_amount_sent") / greatest(col("active_days"), lit(1)), 2))
)

# Composite risk score
risk_scores = (accounts
    .select("account_id", "account_type", "risk_score", "status")
    .join(fraud_agg, "account_id", "left")
    .join(velocity, "account_id", "left")
    .na.fill(0)
    .withColumn("composite_score",
        col("risk_score") * 0.2 +
        col("critical_signals") * 25 +
        col("high_signals") * 10 +
        col("confirmed_fraud") * 50 +
        when(col("max_single_transfer") > 50000, 15).otherwise(0) +
        when(col("unique_recipients") > 20, 10).otherwise(0)
    )
    .withColumn("risk_tier",
        when(col("composite_score") >= 100, "critical")
        .when(col("composite_score") >= 60, "high")
        .when(col("composite_score") >= 30, "medium")
        .otherwise("low")
    )
    .withColumn("_scored_at", current_timestamp())
)

risk_scores.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_risk_scores")
print(f"✅ Risk scores: {spark.table('capstone_gold_risk_scores').count():,} accounts scored")
risk_scores.groupBy("risk_tier").count().orderBy("count", ascending=False).show()

---
# 📋 Section 4: KYC Status Tracking (SCD Type 2)

In [ ]:
%sql
-- KYC SCD2 dimension
DROP TABLE IF EXISTS capstone_dim_kyc_scd2;
CREATE TABLE capstone_dim_kyc_scd2 (
  kyc_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  account_id INT, verification_type STRING, status STRING,
  document_type STRING, risk_level STRING,
  effective_start DATE, effective_end DATE, is_current BOOLEAN
);

-- Initial load
INSERT INTO capstone_dim_kyc_scd2 (account_id, verification_type, status, document_type, risk_level, effective_start, effective_end, is_current)
SELECT account_id, verification_type, status, document_type, risk_level,
  submitted_date AS effective_start, CAST('9999-12-31' AS DATE) AS effective_end, true
FROM kyc_records;

-- Point-in-time query: Was account 100 verified on 2023-06-01?
SELECT * FROM capstone_dim_kyc_scd2
WHERE account_id = 100 AND effective_start <= '2023-06-01' AND effective_end >= '2023-06-01';

---
# 📊 Section 5: Regulatory Reporting Gold Tables

In [ ]:
# SAR (Suspicious Activity Report) data
from pyspark.sql.functions import *

risk = spark.table("capstone_gold_risk_scores")
transfers = spark.table("capstone_silver_wire_transfers")

# Accounts requiring SAR filing (risk_tier = critical or high)
sar_data = (risk
    .filter("risk_tier IN ('critical', 'high')")
    .join(transfers.groupBy(col("from_account").alias("account_id")).agg(
        count("*").alias("recent_transfers"),
        round(sum("amount"), 2).alias("recent_amount")
    ), "account_id", "left")
    .select("account_id", "account_type", "composite_score", "risk_tier",
        "critical_signals", "confirmed_fraud", "recent_transfers", "recent_amount")
    .withColumn("sar_recommended", when(col("composite_score") >= 80, True).otherwise(False))
    .withColumn("_generated_at", current_timestamp())
)
sar_data.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_sar_data")
print(f"✅ SAR data: {spark.table('capstone_gold_sar_data').count():,} accounts flagged")

In [ ]:
# Large Transaction Report (> $10K threshold)
from pyspark.sql.functions import *

transfers = spark.table("capstone_silver_wire_transfers")
large_txn = (transfers
    .filter("amount >= 10000")
    .withColumn("report_date", current_date())
    .select("transfer_id", "from_account", "to_account", "amount", "currency",
        "transfer_date", "status", "report_date")
    .withColumn("_generated_at", current_timestamp())
)
large_txn.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_large_transactions")
print(f"✅ Large transactions (>$10K): {spark.table('capstone_gold_large_transactions').count():,}")

In [ ]:
# ============================================
# 🎯 YOUR TURN: KYC Compliance Dashboard
# ============================================
# Build capstone_gold_kyc_dashboard with:
# 1. Verification status distribution (approved/pending/rejected/expired counts)
# 2. Average days to verify (approved_date - submitted_date)
# 3. Expired verifications count (status = 'expired')
# 4. Risk level distribution
# Write to Delta table
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

kyc = spark.table("kyc_records")
dashboard = (kyc
    .groupBy("status", "risk_level")
    .agg(
        count("*").alias("count"),
        round(avg(datediff(col("approved_date"), col("submitted_date"))), 1).alias("avg_days_to_verify")
    )
    .withColumn("_generated_at", current_timestamp())
)
dashboard.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_kyc_dashboard")
print(f"✅ KYC dashboard: {spark.table('capstone_gold_kyc_dashboard').count():,} rows")
dashboard.show()

---
# 🔔 Section 6: Alerting Logic

In [ ]:
from pyspark.sql.functions import *
from datetime import datetime
import pandas as pd

class FraudAlertEngine:
    """Generate and deduplicate fraud alerts."""
    
    def __init__(self):
        self.alerts = []
    
    def check_risk_threshold(self, risk_df, threshold=80):
        """Alert on accounts crossing risk threshold."""
        high_risk = risk_df.filter(f"composite_score >= {threshold}").collect()
        for row in high_risk:
            self.alerts.append({
                "alert_type": "risk_threshold_breach",
                "account_id": row.account_id,
                "severity": "critical" if row.composite_score >= 100 else "high",
                "details": f"Score: {row.composite_score:.0f}, Tier: {row.risk_tier}",
                "generated_at": datetime.now().isoformat()
            })
    
    def check_large_transactions(self, transfers_df, threshold=50000):
        """Alert on unusually large transactions."""
        large = transfers_df.filter(f"amount >= {threshold}").collect()
        for row in large[:20]:  # Limit for demo
            self.alerts.append({
                "alert_type": "large_transaction",
                "account_id": row.from_account,
                "severity": "high",
                "details": f"Amount: ${row.amount:,.2f}, To: {row.to_account}",
                "generated_at": datetime.now().isoformat()
            })
    
    def deduplicate(self):
        """Remove duplicate alerts (same type + account)."""
        seen = set()
        unique = []
        for alert in self.alerts:
            key = (alert["alert_type"], alert["account_id"])
            if key not in seen:
                seen.add(key)
                unique.append(alert)
        self.alerts = unique
    
    def publish(self):
        """Write alerts to Delta table."""
        self.deduplicate()
        if self.alerts:
            pdf = pd.DataFrame(self.alerts)
            spark.createDataFrame(pdf).write.format("delta").mode("overwrite").saveAsTable("capstone_alerts")
            print(f"🔔 Published {len(self.alerts)} alerts")
            for a in self.alerts[:5]:
                icon = "🔴" if a["severity"] == "critical" else "🟠"
                print(f"  {icon} [{a['severity']}] {a['alert_type']}: account {a['account_id']}")
        else:
            print("  No alerts generated")

# Run alert engine
engine = FraudAlertEngine()
engine.check_risk_threshold(spark.table("capstone_gold_risk_scores"), threshold=80)
engine.check_large_transactions(spark.table("capstone_silver_wire_transfers"), threshold=50000)
engine.publish()

---
# 📜 Section 7: Audit & Compliance

In [ ]:
%sql
-- Audit log table
DROP TABLE IF EXISTS capstone_audit_log;
CREATE TABLE capstone_audit_log (
  audit_id BIGINT GENERATED ALWAYS AS IDENTITY,
  event_type STRING, table_name STRING, operation STRING,
  record_count INT, user_name STRING, event_timestamp TIMESTAMP,
  batch_id STRING, details STRING
);

-- Log our pipeline execution
INSERT INTO capstone_audit_log (event_type, table_name, operation, record_count, user_name, event_timestamp, batch_id, details)
VALUES
  ('pipeline_run', 'capstone_silver_wire_transfers', 'CDC_MERGE', 0, 'pipeline_service', current_timestamp(), 'batch_capstone', 'CDC processing complete'),
  ('pipeline_run', 'capstone_gold_risk_scores', 'OVERWRITE', 0, 'pipeline_service', current_timestamp(), 'batch_capstone', 'Risk scoring complete'),
  ('alert_generated', 'capstone_alerts', 'INSERT', 0, 'alert_engine', current_timestamp(), 'batch_capstone', 'Fraud alerts published');

SELECT * FROM capstone_audit_log ORDER BY event_timestamp DESC;

In [ ]:
%sql
-- Data retention: demonstrate VACUUM alignment with compliance
-- Transaction data: 7 years retention → VACUUM RETAIN 2555 HOURS (7*365*24)
-- For demo, just show the concept:
DESCRIBE HISTORY capstone_silver_wire_transfers LIMIT 5;

---
# ✅ Section 8: Final Verification

In [ ]:
%sql
SELECT 'capstone_silver_wire_transfers' AS tbl, COUNT(*) AS rows FROM capstone_silver_wire_transfers
UNION ALL SELECT 'capstone_silver_compliance', COUNT(*) FROM capstone_silver_compliance
UNION ALL SELECT 'capstone_gold_risk_scores', COUNT(*) FROM capstone_gold_risk_scores
UNION ALL SELECT 'capstone_gold_sar_data', COUNT(*) FROM capstone_gold_sar_data
UNION ALL SELECT 'capstone_gold_large_transactions', COUNT(*) FROM capstone_gold_large_transactions
UNION ALL SELECT 'capstone_gold_kyc_dashboard', COUNT(*) FROM capstone_gold_kyc_dashboard
UNION ALL SELECT 'capstone_dim_kyc_scd2', COUNT(*) FROM capstone_dim_kyc_scd2
UNION ALL SELECT 'capstone_alerts', COUNT(*) FROM capstone_alerts
UNION ALL SELECT 'capstone_audit_log', COUNT(*) FROM capstone_audit_log
ORDER BY tbl;

---
# 📗 Enhancement: FinTech Pipeline -- Production Patterns

## Architecture Overview

```
Source Systems
    |
    v
Bronze Layer (raw, append-only)
    |
    v
Silver Layer (cleaned, validated, deduplicated)
    |
    v
Gold Layer (aggregated, business-ready)
    |
    v
Serving (SQL Warehouse, APIs, ML)
```

## Key Tables: transactions, accounts, customers, fraud_signals

## Production Checklist

| Item | Status |
|------|--------|
| Idempotent pipeline (safe to rerun) | Required |
| Data quality checks at each layer | Required |
| Incremental processing (not full refresh) | Required |
| Error handling and dead letter queue | Required |
| Monitoring and alerting | Required |
| Documentation and data catalog | Required |

## Common Patterns Used

1. **Auto Loader** for file ingestion
2. **MERGE** for CDC and upserts
3. **ROW_NUMBER** for deduplication
4. **Window functions** for running totals, rankings
5. **DLT expectations** for quality gates
6. **Lakeflow Jobs** for orchestration

In [ ]:
# FinTech Pipeline -- Key SQL Patterns
print("FinTech Pipeline -- Production SQL Patterns")
print("=" * 60)

# Pattern 1: Incremental load with watermark
print("\n1. Incremental load pattern:")
print("""
    MERGE INTO silver.transactions AS target
    USING (
        SELECT * FROM bronze.raw_transactions
        WHERE _ingested_at > (SELECT MAX(_ingested_at) FROM silver.transactions)
    ) AS source
    ON target.id = source.id
    WHEN MATCHED AND source._row_hash != target._row_hash THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Pattern 2: Deduplication
print("2. Deduplication pattern:")
print("""
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _ingested_at DESC
            ) AS rn
        FROM bronze.raw_transactions
    ) WHERE rn = 1
""")

# Pattern 3: Quality checks
print("3. Quality check pattern:")
quality_checks = [
    f"SELECT COUNT(*) FROM silver.transactions WHERE id IS NULL",
    f"SELECT COUNT(*) - COUNT(DISTINCT id) FROM silver.transactions",
    f"SELECT COUNT(*) FROM silver.transactions WHERE amount < 0",
]
for qc in quality_checks:
    print(f"   {qc}")


---
# Enhancement: FinTech Pipeline -- Production Patterns

## Architecture

```
Source: transactions, accounts, fraud_signals
    |
    v (Auto Loader / Kafka)
Bronze Layer (raw, append-only, Delta)
    |
    v (DLT with expectations)
Silver Layer (cleaned, validated, typed)
    |
    v (Spark / dbt)
Gold Layer (aggregated, business-ready)
    |
    v
Serving (SQL Warehouse, APIs, ML)
```

## Key Tables: transactions, accounts, customers

## Production Checklist

| Item | Requirement |
|------|-------------|
| Idempotent pipeline | Safe to rerun for any date |
| Quality checks | At each layer boundary |
| Incremental processing | Not full refresh |
| Error handling | DLQ for bad records |
| Monitoring | Alerts on failure |
| Documentation | Data catalog entries |

## Common Patterns Used

1. **Auto Loader** for file ingestion (cloudFiles format)
2. **MERGE** for CDC and upserts
3. **ROW_NUMBER** for deduplication
4. **Window functions** for running totals, rankings
5. **DLT expectations** for quality gates
6. **Lakeflow Jobs** for orchestration

In [ ]:
# FinTech Pipeline -- Key SQL Patterns
print("FinTech Pipeline -- Production SQL Patterns")
print("=" * 60)

patterns = {
    "Incremental load (watermark)": f"""
    MERGE INTO silver.transactions AS target
    USING (
        SELECT * FROM bronze.raw_transactions
        WHERE _ingested_at > (SELECT COALESCE(MAX(_ingested_at), '1900-01-01')
                              FROM silver.transactions)
    ) AS source
    ON target.id = source.id
    WHEN MATCHED AND source._row_hash != target._row_hash THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """,

    "Deduplication (keep latest)": f"""
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _ingested_at DESC
            ) AS rn
        FROM bronze.raw_transactions
    ) WHERE rn = 1
    """,

    "Quality checks": f"""
    SELECT 'null_id' AS check_name, COUNT(*) AS failures
    FROM silver.transactions WHERE id IS NULL
    UNION ALL
    SELECT 'duplicate_id', COUNT(*) - COUNT(DISTINCT id)
    FROM silver.transactions
    UNION ALL
    SELECT 'negative_amount', COUNT(*)
    FROM silver.transactions WHERE amount < 0
    """,
}

for pname, sql in patterns.items():
    print(f"\n" + pname + ":")
    print(sql)


---
# 🎓 Capstone Complete!

**What was built:**
- CDC processing pipeline (MERGE for wire_transfers with I/U/D)
- Fraud scoring engine (composite risk scores, tier assignment)
- KYC SCD Type 2 dimension (point-in-time queries)
- 4 regulatory Gold tables (SAR, large transactions, KYC dashboard, risk scores)
- Alert engine with deduplication
- Audit trail table
- Data retention awareness

**Compliance features:**
- All changes audited
- Point-in-time query capability
- Data classification documented
- Retention policies defined
- PII handling noted

**Next:** Notebook 23 — Capstone: IoT Pipeline

---